# Notebook 20 – Mini Assessment
### Sprint 7 – Machine Learning Fundamentals for AI/ML Engineers

This notebook contains conceptual questions with model answers, followed by short coding exercises with
solutions.

## Part A — Conceptual Questions

**Q1. What is Machine Learning?**
A field of AI where systems learn patterns directly from data (rather than explicit rules) to make predictions or
decisions on new, unseen data.

**Q2. Supervised vs Unsupervised Learning.**
Supervised uses labeled data (X and known y) to learn a mapping and predict y for new X. Unsupervised works with
unlabeled data (X only) to discover structure — no target to predict.

**Q3. Classification vs Regression.**
Classification predicts a discrete category; Regression predicts a continuous numeric value. Same overall
workflow, different target type and evaluation metrics.

**Q4. What is a feature?**
An individual measurable input variable used to make predictions (independent variable/predictor).

**Q5. What is a target?**
The variable the model tries to predict (dependent variable/label).

**Q6. Why split data into train and test?**
To get an honest estimate of performance on unseen data. Evaluating only on training data lets a model "cheat" by
memorizing, hiding overfitting completely.

**Q7. What is overfitting?**
Learning training data too well (including noise), resulting in low training error but high error on new data.

**Q8. What is underfitting?**
Model too simple to capture the real pattern, resulting in poor performance on BOTH training and test data.

**Q9. Explain Bias-Variance Tradeoff.**
`Total Error = Bias² + Variance + Irreducible Error`. Bias = error from oversimplifying (underfitting). Variance =
error from oversensitivity to the training sample (overfitting). Increasing complexity typically lowers bias but
raises variance; the goal is minimizing their combined effect.

**Q10. Explain Linear Regression.**
Fits a straight-line relationship between features and a continuous target by minimizing sum of squared residuals
(Least Squares). Assumes a linear relationship.

**Q11. Explain Logistic Regression.**
Fits a linear combination of features passed through a sigmoid function to output a probability (0-1), then
applies a threshold (commonly 0.5) to assign a class.

**Q12. Why is Logistic Regression used for classification (despite its name)?**
"Regression" refers to how it's fit internally (regressing a continuous log-odds score) — its final output, after
the sigmoid and threshold, is a discrete class label, making it a classification algorithm in practice.

**Q13. Explain Decision Trees.**
Repeatedly splits data on feature thresholds (choosing splits that most reduce impurity — Gini or Entropy) until
reaching leaf nodes that output a prediction. Easy to interpret by tracing root-to-leaf.

**Q14. Gini vs Entropy.**
Both measure node impurity. Gini: `1 - Σp_i²`, simpler/faster. Entropy: `-Σp_i*log2(p_i)`, information-theory
based, slightly costlier. Usually produce similar trees; Gini is sklearn's default.

**Q15. Decision Tree vs Random Forest.**
A single tree is fast/interpretable but high-variance. Random Forest trains many trees on bootstrap-sampled rows
and random feature subsets, averaging/voting — trading interpretability for lower variance.

**Q16. Explain KNN.**
A lazy-learning algorithm with no real training phase — predicts a new point by finding the K closest training
points and taking a majority vote (classification) or average (regression).

**Q17. Why does KNN require feature scaling?**
It relies on distance calculations. Unscaled features with larger ranges dominate the distance metric — same
reasoning applies to K-Means and SVM.

**Q18. Explain SVM.**
Finds the hyperplane separating classes with the maximum margin, using only the closest points (support vectors)
to define it. Non-linear boundaries handled via the kernel trick (e.g., RBF).

**Q19. Explain K-Means.**
Unsupervised clustering: assigns points to the nearest of K centroids, then updates centroids to the mean of
assigned points, repeating until convergence. Requires K chosen in advance and feature scaling.

**Q20. Accuracy vs Precision vs Recall.**
Accuracy = overall fraction correct (misleading on imbalanced data). Precision = of predicted positives, how many
were correct. Recall = of actual positives, how many were caught. Which to prioritize depends on which error
type is more costly.

**Q21. What is F1 Score?**
Harmonic mean of precision and recall: `F1 = 2*(P*R)/(P+R)`. A single balanced number for comparing models.

**Q22. What is Cross Validation?**
Repeatedly splitting training data into K folds, training on K-1 and validating on the remainder, then averaging
— giving a more reliable estimate (and a stability measure via std dev) than a single split.

**Q23. How would you select the best model?**
Not by highest single-metric score alone. Compare cross-validated performance on the priority metric, check the
train-validation gap for overfitting, check stability (CV std dev), and only then use simplicity/speed/
interpretability to break ties — exactly the process followed in Notebooks 17-19.

## Part B — Coding Exercises

### Exercise 1
Load the Breast Cancer Wisconsin dataset, split it into train/test (80/20, stratified, random_state=42), and
print the shape of each resulting set.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import pandas as pd

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (455, 30)
X_test:  (114, 30)
y_train: (455,)
y_test:  (114,)


### Exercise 2
Train a Logistic Regression model on the split above, and print its test-set accuracy and recall on the
malignant class (label 0).

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, recall_score

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=42))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Recall (malignant):", round(recall_score(y_test, y_pred, pos_label=0), 4))

Accuracy: 0.9825
Recall (malignant): 0.9762


### Exercise 3
Using 5-fold Stratified Cross Validation on the training set, compute the mean and standard deviation of accuracy
for the model from Exercise 2.

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")

print("Fold scores:", scores.round(4))
print("Mean CV Accuracy:", round(scores.mean(), 4))
print("Std Dev:", round(scores.std(), 4))

Fold scores: [0.967 0.989 0.978 0.989 0.967]
Mean CV Accuracy: 0.978
Std Dev: 0.0098


### Exercise 4
Write a small function `bias_variance_label(train_score, test_score, gap_threshold=0.05)` that returns
"underfitting", "overfitting", or "good fit" given a train and test score. Test it on 3 example score pairs.

In [4]:
def bias_variance_label(train_score, test_score, gap_threshold=0.05, low_perf_threshold=0.80):
    gap = train_score - test_score
    if train_score < low_perf_threshold and test_score < low_perf_threshold:
        return "underfitting"
    elif gap > gap_threshold:
        return "overfitting"
    else:
        return "good fit"

examples = [
    (0.65, 0.63),   # both low, close together
    (1.00, 0.80),   # train much higher than test
    (0.95, 0.93),   # both high, close together
]

for train_s, test_s in examples:
    label = bias_variance_label(train_s, test_s)
    print(f"train={train_s}, test={test_s} -> {label}")

train=0.65, test=0.63 -> underfitting
train=1.0, test=0.8 -> overfitting
train=0.95, test=0.93 -> good fit


### Exercise 5
Given a confusion matrix for a binary classifier (2x2 numpy array, [[TN, FP], [FN, TP]]), write a function that
manually computes precision, recall, and F1 (without sklearn's metric functions), then verify it matches
sklearn's output on the Exercise 2 model's test predictions.

In [5]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

def manual_metrics(cm):
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])  # rows/cols ordered [malignant, benign]
cm_pos1 = np.array([[cm[0,0], cm[0,1]], [cm[1,0], cm[1,1]]])  # benign(1) treated as "positive" here

manual_p, manual_r, manual_f1 = manual_metrics(cm_pos1)
sklearn_p = precision_score(y_test, y_pred, pos_label=1)
sklearn_r = recall_score(y_test, y_pred, pos_label=1)
sklearn_f1 = f1_score(y_test, y_pred, pos_label=1)

print(f"Manual  -> Precision: {manual_p:.4f}  Recall: {manual_r:.4f}  F1: {manual_f1:.4f}")
print(f"sklearn -> Precision: {sklearn_p:.4f}  Recall: {sklearn_r:.4f}  F1: {sklearn_f1:.4f}")
print("Match:", np.isclose(manual_p, sklearn_p) and np.isclose(manual_r, sklearn_r) and np.isclose(manual_f1, sklearn_f1))

Manual  -> Precision: 0.9861  Recall: 0.9861  F1: 0.9861
sklearn -> Precision: 0.9861  Recall: 0.9861  F1: 0.9861
Match: True
